# Wenn das Modell versagt

**Einstiegsfrage:** Ein Modell sagt: Ein solcher Tag kommt alle 13'800 Jahre. Dann passiert er zweimal in zehn Jahren. Wer haftet?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2027)


def pruefe(name, wert, soll, toleranz):
    """Selbstkontrolle: vergleicht Ihren Wert mit dem erwarteten Bereich."""
    if wert is None:
        print(f"{name}: noch nicht bearbeitet")
    elif abs(wert - soll) <= toleranz:
        print(f"{name}: passt ({wert:.4g})")
    else:
        print(f"{name}: prüfen Sie Ihren Code, erwartet wird etwa {soll:.4g}, Sie haben {wert:.4g}")

## Aufgabe 1: Dicke Ränder
Erzeugen Sie 2520 Tage mal 2000 Läufe mit Tagesvolatilität 1 %: einmal normalverteilt, einmal t-verteilt mit 3 Freiheitsgraden.

*Hilfestellung:* `rng.standard_t(3, size=...)` hat die Standardabweichung $\sqrt{3}$. Teilen Sie durch diesen Wert und multiplizieren Sie mit 0.01.

In [ ]:
form = (2000, 2520)
normal = dick = None
normal = rng.normal(0, 0.01, size=form)
dick = rng.standard_t(3, size=form) * 0.01 / np.sqrt(3)
t5 = None if dick is None else np.mean(np.sum(dick <= -0.05, axis=1))
n5 = None if normal is None else np.mean(np.sum(normal <= -0.05, axis=1))
pruefe('Tage bis -5 %, normal', n5, 0.0, 0.01)
pruefe('Tage bis -5 %, dick', t5, 4.0, 0.3)

## Aufgabe 2: Value at Risk und Expected Shortfall
Der VaR 99 % ist das 1-%-Perzentil. Der Expected Shortfall ist der Durchschnitt aller Tage, die noch schlechter sind. Welche Zahl trennt die beiden Modelle besser?

In [ ]:
es_normal = es_dick = None
v1 = np.percentile(normal, 1)
v2 = np.percentile(dick, 1)
es_normal = normal[normal <= v1].mean()
es_dick = dick[dick <= v2].mean()
pruefe('ES normal', es_normal, -0.0266, 0.0005)
pruefe('ES dick', es_dick, -0.0404, 0.002)

## Challenge: Schulden als Verstärker
Ein Fonds hat Tagesvolatilität 0.5 %, Tagesdrift 0.04 % und arbeitet mit Hebel $h$: Die Tagesrendite des Eigenkapitals ist $h \cdot r$. Wie oft fällt das Eigenkapital innerhalb eines Jahres (252 Tage) um mindestens 80 %? Vergleichen Sie Hebel 5, 10 und 25, normal und dick.

*Hilfestellung:* Rechnen Sie mit `np.log1p(np.maximum(h * r, -0.999999))`, summieren Sie mit `np.cumsum` und vergleichen Sie mit `np.log(0.2)`.

In [ ]:
def ruin(r, h):
    wert = None
    lw = np.cumsum(np.log1p(np.maximum(h * r, -0.999999)), axis=1)
    wert = np.mean(np.any(lw <= np.log(0.2), axis=1))
    return wert


r = 0.0004 + rng.standard_t(3, size=(100_000, 252)) * 0.005 / np.sqrt(3)
pruefe('Dick, Hebel 10', ruin(r, 10), 0.0189, 0.003)
pruefe('Dick, Hebel 25', ruin(r, 25), 0.336, 0.01)